In [9]:
import subprocess

import numpy as np
import pandas as pd

In [13]:
output = subprocess.run(
    f"mysql -t -u employee_user -p123 < test_employees_sha.sql",
    cwd="./test_db",
    capture_output=True,
    text=True,
    shell=True,
).stdout.split("\n")

In [38]:
expected = dict()
found = dict()
match = dict()

output_clean = []
for row in output:
    if row.startswith("+"):
        continue
    mod_row = []
    for col in row.strip().split("|"):
        if col == "":
            continue
        mod_row.append(col.strip())

    if len(mod_row) < 3:
        continue

    output_clean.append(mod_row)

table_section = None
for row in output_clean:
    if (
        row[1].startswith("expected")
        and row[2].startswith("expected")
        and table_section != "expected"
    ):
        table_section = "expected"
        continue
    elif (
        row[1].startswith("found")
        and row[2].startswith("found")
        and table_section != "found"
    ):
        table_section = "found"
        continue
    elif (
        row[1].endswith("match")
        and row[2].endswith("match")
        and table_section != "match"
    ):
        table_section = "match"
        continue

    match table_section:
        case "expected":
            expected[row[0]] = [int(row[1]), row[2]]
        case "found":
            found[row[0]] = [int(row[1]), row[2]]
        case "match":
            match[row[0]] = [row[1].upper() == "OK", row[2].upper() == "OK"]

all_match = bool(np.all(list(match.values())))

record_df = pd.DataFrame(
    {
        "table_name": list(found.keys()),
        "expected_records": np.array(list(expected.values()))[:, 0].tolist(),
        "found_records": np.array(list(found.values()))[:, 0].tolist(),
        "records_match": np.array(list(match.values()))[:, 0].tolist(),
    }
)

crc_df = pd.DataFrame(
    {
        "table_name": list(found.keys()),
        "expected_crc": np.array(list(expected.values()))[:, 1].tolist(),
        "found_crc": np.array(list(found.values()))[:, 1].tolist(),
        "crc_match": np.array(list(match.values()))[:, 1].tolist(),
    }
)

if not all_match:
    raise AssertionError(
        "The Database Integrity test failed!\n\n"
        "-----------------------------------\n"
        "----------- TEST OUTPUT -----------\n"
        f"-----------------------------------\n\n{record_df}\n\n{crc_df}"
    )

print(
    "-----------------------------------\n"
    "----------- TEST OUTPUT -----------\n"
    f"-----------------------------------\n\n{record_df}\n\n{crc_df}"
)

AssertionError: The Database Integrity test failed!

-----------------------------------
----------- TEST OUTPUT -----------
-----------------------------------

     table_name expected_records found_records  records_match
0   departments                9             9           True
1      dept_emp           331603        331603           True
2  dept_manager               24            24           True
3     employees           300024        300024           True
4      salaries          2844047       2844047           True
5        titles           443308        443308           True

     table_name                              expected_crc  \
0   departments  4b315afa0e35ca6649df897b958345bcb3d2b764   
1      dept_emp  d95ab9fe07df0865f592574b3b33b9c741d9fd1b   
2  dept_manager  9687a7d6f93ca8847388a42a6d8d93982a841c6c   
3     employees  4d4aa689914d8fd41db7e45c2168e7dcb9697359   
4      salaries  b5a1785c27d75e33a4173aaa22ccf41ebd7d4a9f   
5        titles  d12d5f746b88f07e69b9

,table_name,expected_records,found_records,records_match,expected_crc,found_crc,crc_match
0,departments,9,9,True,4b315afa0e35ca6649df897b958345bcb3d2b764,e66fb1044ebd3d9f7567c29825c02669b0a08156,False
1,dept_emp,331603,331603,True,d95ab9fe07df0865f592574b3b33b9c741d9fd1b,d95ab9fe07df0865f592574b3b33b9c741d9fd1b,True
2,dept_manager,24,24,True,9687a7d6f93ca8847388a42a6d8d93982a841c6c,9687a7d6f93ca8847388a42a6d8d93982a841c6c,True
3,employees,300024,300024,True,4d4aa689914d8fd41db7e45c2168e7dcb9697359,4d4aa689914d8fd41db7e45c2168e7dcb9697359,True
4,salaries,2844047,2844047,True,b5a1785c27d75e33a4173aaa22ccf41ebd7d4a9f,b5a1785c27d75e33a4173aaa22ccf41ebd7d4a9f,True
5,titles,443308,443308,True,d12d5f746b88f07e69b9e36675b6067abb01b60e,d12d5f746b88f07e69b9e36675b6067abb01b60e,True


array([['9', '4b315afa0e35ca6649df897b958345bcb3d2b764'],
       ['331603', 'd95ab9fe07df0865f592574b3b33b9c741d9fd1b'],
       ['24', '9687a7d6f93ca8847388a42a6d8d93982a841c6c'],
       ['300024', '4d4aa689914d8fd41db7e45c2168e7dcb9697359'],
       ['2844047', 'b5a1785c27d75e33a4173aaa22ccf41ebd7d4a9f'],
       ['443308', 'd12d5f746b88f07e69b9e36675b6067abb01b60e']],
      dtype='<U40')